In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Math

# ==============================================================================
# PROBLEM: Pole-Zero Plots & Exact ROC Determination for Multiple Z-Transforms
# ==============================================================================

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Solution Overview (Pole-Zero Maps & Exact ROC Analysis)</b><br>
* <b>X₁(z):</b> Pole at z = -2, Zero at z = 0.5. ROC: |z| < 2 (Anti-causal signal).<br>
* <b>X₂(z):</b> Poles at -0.5 and 2/3. Causal signal -> ROC: |z| > 2/3.<br>
* <b>X₃(z):</b> Two-sided signal -> Annular ROC: 2/3 < |z| < 1.5.<br>
* <b>Note:</b> Use the dropdown menu below to select and visualize each transformation interactively.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

def plot_z_transforms(system_choice):
    with out:
        clear_output(wait=True)
        
        fig, ax_pz = plt.subplots(figsize=(8, 6))
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-3.0, 3.0)
        ax_pz.set_ylim(-3.0, 3.0)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)

        theta = np.linspace(0, 2*np.pi, 200)
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5, label='Unit Circle')

        x_vals = np.linspace(-3.5, 3.5, 400)
        y_vals = np.linspace(-3.5, 3.5, 400)
        X, Y = np.meshgrid(x_vals, y_vals)
        Z_dist = np.sqrt(X**2 + Y**2)

        if system_choice.startswith("X1"):
            zeros_x, zeros_y = [0.5], [0]
            poles_x, poles_y = [-2.0], [0]
            roc_mask = Z_dist < 2.0
            ax_pz.imshow(roc_mask, extent=(-3.5, 3.5, -3.5, 3.5), origin='lower', cmap='Greens', alpha=0.25, zorder=0)
            ax_pz.plot(2.0 * np.cos(theta), 2.0 * np.sin(theta), 'g:', linewidth=2, label='ROC Boundary (|z| = 2)')
            title_str = "X₁(z): Pole at z = -2, Zero at z = 0.5 (ROC: |z| < 2)"

        elif system_choice.startswith("X2"):
            zeros_x, zeros_y = [1/3], [0]
            poles_x, poles_y = [-0.5, 2/3], [0, 0]
            roc_mask = Z_dist > (2/3)
            ax_pz.imshow(roc_mask, extent=(-3.5, 3.5, -3.5, 3.5), origin='lower', cmap='Greens', alpha=0.25, zorder=0)
            ax_pz.plot((2/3) * np.cos(theta), (2/3) * np.sin(theta), 'g:', linewidth=2, label='ROC Boundary (|z| = 2/3)')
            title_str = "X₂(z): Causal Signal (ROC: |z| > 2/3)"

        elif system_choice.startswith("X3"):
            zeros_roots = np.roots([1, 1, -2]) 
            zeros_x, zeros_y = zeros_roots.real, zeros_roots.imag
            
            inner_r = 2/3
            outer_r = 1.5
            poles_x, poles_y = [inner_r, outer_r], [0, 0]
            
            roc_mask = (Z_dist > inner_r) & (Z_dist < outer_r)
            ax_pz.imshow(roc_mask, extent=(-3.5, 3.5, -3.5, 3.5), origin='lower', cmap='Greens', alpha=0.25, zorder=0)
            ax_pz.plot(inner_r * np.cos(theta), inner_r * np.sin(theta), 'g:', linewidth=2, label=f'Inner ROC (|z| = {inner_r:.2f})')
            ax_pz.plot(outer_r * np.cos(theta), outer_r * np.sin(theta), 'g:', linewidth=2, label=f'Outer ROC (|z| = {outer_r:.2f})')
            title_str = f"X₃(z): Two-sided Signal (ROC: {inner_r:.2f} < |z| < {outer_r:.2f})"

        # Plot Zeros and Poles
        ax_pz.scatter(zeros_x, zeros_y, s=120, facecolors='none', edgecolors='b', linewidths=2, marker='o', label='Zeros')
        ax_pz.scatter(poles_x, poles_y, s=140, color='purple', marker='x', linewidths=3, label='Poles')

        ax_pz.set_title(title_str, fontsize=10, fontweight='bold')
        ax_pz.set_xlabel('Real Part', fontsize=9)
        ax_pz.set_ylabel('Imaginary Part', fontsize=9)

        unit_circle_handle = plt.Line2D([0], [0], color='k', linestyle='--', alpha=0.5, label='Unit Circle')
        pole_handle = plt.Line2D([0], [0], marker='x', color='purple', markersize=8, markeredgewidth=3, linestyle='None', label='Poles')
        zero_handle = plt.Line2D([0], [0], marker='o', markerfacecolor='none', markeredgecolor='b', markersize=8, markeredgewidth=2, linestyle='None', label='Zeros')
        
        ax_pz.legend(handles=[unit_circle_handle, pole_handle, zero_handle], loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8)

        plt.show()

        print("-" * 115)
        print("GEOMETRIC & ROC INTERPRETATION:")
        print("1. Poles strictly define the boundary limits of the Region of Convergence (ROC).")
        print("2. For anti-causal configurations (like X1), the ROC spans inward: |z| < r_pole.")
        print("3. For causal configurations (like X2), the ROC spans outward: |z| > r_pole.")
        print("4. For two-sided configurations (like X3), the ROC forms an annular ring between poles.")
        print("-" * 115)

# Dropdown menu widget with increased width layout
dropdown = widgets.Dropdown(
    options=[
        "X1(z) = (1 - 0.5z^-1) / (1 + 2z^-1)",
        "X2(z) = [1 - (1/3)z^-1] / [(1 + 0.5z^-1)(1 - (2/3)z^-1)]",
        "X3(z) = (1 + z^-1 - 2z^-2) / (1 - (13/6)z^-1 + z^-2)"
    ],
    value="X1(z) = (1 - 0.5z^-1) / (1 + 2z^-1)",
    description='System:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

plot_z_transforms(dropdown.value)

interactive_plot = widgets.interactive(plot_z_transforms, system_choice=dropdown)
display(widgets.VBox([interactive_plot, out]))